# Prithvi WxC Downscaling with ECCC Data: Model Fine-tuning

This notebook is a walk through to use the `Prithvi` downscaling model for fine-tuning using the `ECCC` data

We show how to initalize the model, load the pretrained weights, and finetune it

To replicate the results show in this notebook please download the required files from our [Hugging Face](https://huggingface.co/ibm-granite/granite-geospatial-wxc-downscaling) repository

You need `git lfs` installed to download large files

**This notebook is a simple plug-and-play example** 

We provide only **1 data sample**. See `./examples/eccc_downscaling/notebooks/README.md` to download and preprocess the remaining files

---

## Setup

Python >= 3.10 is required

Make sure that your current working directory is `granite-wxc/`

In [ ]:
!pwd

If your current directory is not `granite-wxc/`, change it using the following command:

```bash
%cd <local>/granite-wxc/
```

Replace `<local>` with the appropriate path prefix 

In [ ]:
!pip install -q git+https://github.com/NASA-IMPACT/Prithvi-WxC.git

In [ ]:
!pip install -q h5netcdf matplotlib wget pyyaml xarray scipy torch tqdm pysteps cartopy


In [ ]:
import logging
import warnings
logging.disable(logging.CRITICAL)
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
# ===================== CONFIGURATION / HARDWARE (EDIT ME) =====================
import os
import subprocess

enable_gpu_auto_select = True
min_free_mem_gb = 10.0
prefer_idle_gpus = True
max_gpus = 4
force_visible_devices = None  # e.g. "0,1"
seed = 123


def _parse_device_list(value):
    if value is None:
        return []
    return [int(part.strip()) for part in str(value).split(",") if part.strip()]


def _query_gpu_stats():
    query = "index,memory.total,memory.used,utilization.gpu"
    try:
        out = subprocess.check_output(
            ["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader,nounits"],
            text=True,
        )
    except Exception as exc:
        print(f"GPU status query failed (nvidia-smi unavailable): {exc}")
        return []

    stats = []
    for raw in out.strip().splitlines():
        if not raw.strip():
            continue
        idx_str, total_str, used_str, util_str = [part.strip() for part in raw.split(",")]
        util_digits = "".join(ch for ch in util_str if ch.isdigit())
        total_mb = int(total_str)
        used_mb = int(used_str)
        util = int(util_digits) if util_digits else 100
        stats.append(
            {
                "index": int(idx_str),
                "total_mb": total_mb,
                "used_mb": used_mb,
                "free_mb": max(total_mb - used_mb, 0),
                "util": util,
            }
        )
    return stats


def _select_gpu_indices(gpu_stats, threshold_mb, prefer_idle, max_devices):
    ranked = sorted(
        gpu_stats,
        key=(lambda g: (-g["free_mb"], g["util"], g["used_mb"], g["index"]))
        if prefer_idle
        else (lambda g: g["index"]),
    )
    selected = [g["index"] for g in ranked if g["free_mb"] >= threshold_mb]
    if max_devices is not None:
        selected = selected[: int(max_devices)]
    return selected


def configure_gpu_hardware():
    gpu_stats = _query_gpu_stats()
    available = [g["index"] for g in gpu_stats]
    threshold_mb = int(float(min_free_mem_gb) * 1024)
    selected = []
    reason = ""
    forced = force_visible_devices is not None and str(force_visible_devices).strip() != ""

    if forced:
        os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
        os.environ["CUDA_VISIBLE_DEVICES"] = str(force_visible_devices)
        selected = _parse_device_list(force_visible_devices)
        reason = "force_visible_devices override applied."
    elif enable_gpu_auto_select and gpu_stats:
        selected = _select_gpu_indices(gpu_stats, threshold_mb, prefer_idle_gpus, max_gpus)
        if not selected:
            selected = available[:]
            reason = (
                f"No GPU met min_free_mem_gb={min_free_mem_gb}. "
                "Falling back to all detected GPUs."
            )
        else:
            reason = "Auto-selected GPUs by free memory/utilization."
        os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
        os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(i) for i in selected)
    elif gpu_stats:
        selected = available[:] if max_gpus is None else available[: int(max_gpus)]
        os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
        os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(i) for i in selected)
        reason = "Auto-select disabled; using detected GPUs."
    else:
        reason = "No GPUs detected from nvidia-smi."

    if not gpu_stats:
        try:
            import torch  # fallback when nvidia-smi is unavailable

            count = torch.cuda.device_count()
        except Exception:
            count = 0
        if forced and selected:
            count = max(count, len(selected))
        if count > 0 and not forced:
            selected = list(range(count))
            os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
            os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(i) for i in selected)
            reason = "nvidia-smi unavailable; using all torch-detected GPUs."
        elif count == 0 and not forced:
            selected = []

    if forced and selected and available:
        missing = [idx for idx in selected if idx not in available]
        if missing:
            print(f"Warning: forced GPU indices not detected by nvidia-smi: {missing}")

    selected_stats = {g["index"]: g for g in gpu_stats}
    use_gpu = len(selected) > 0
    world_size = len(selected) if use_gpu else 0
    strategy = "data_parallel" if world_size > 1 else ("single_gpu" if world_size == 1 else "cpu")

    print("Hardware summary:")
    print(f"  selected_original_gpus={selected}")
    print(f"  selected_visible_gpus={list(range(world_size)) if use_gpu else []}")
    if "CUDA_VISIBLE_DEVICES" in os.environ:
        print(f"  CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']}")
    if selected:
        for visible_idx, original_idx in enumerate(selected):
            stat = selected_stats.get(original_idx)
            if stat is None:
                print(f"  GPU orig={original_idx} vis={visible_idx}: stats unavailable")
                continue
            print(
                "  GPU orig={orig} vis={vis}: total={total:.1f}GB used={used:.1f}GB free={free:.1f}GB util={util}%".format(
                    orig=original_idx,
                    vis=visible_idx,
                    total=stat["total_mb"] / 1024.0,
                    used=stat["used_mb"] / 1024.0,
                    free=stat["free_mb"] / 1024.0,
                    util=stat["util"],
                )
            )
    else:
        print("  Warning: no usable GPUs selected; running on CPU.")
    print(f"  world_size={world_size}")
    print(f"  strategy={strategy}")
    if reason:
        print(f"  note={reason}")

    return {
        "selected_original_indices": selected,
        "world_size": world_size,
        "use_gpu": use_gpu,
        "strategy": strategy,
    }


GPU_SELECTION = configure_gpu_hardware()
SELECTED_GPU_ORIGINAL_IDS = GPU_SELECTION["selected_original_indices"]
SELECTED_GPU_COUNT = GPU_SELECTION["world_size"]
SELECTED_USE_GPU = GPU_SELECTION["use_gpu"]


## GPU Selection Notes

- Set `force_visible_devices` (for example `"1"` or `"0,2"`) to bypass auto-selection and use those original GPU IDs.
- With auto-selection enabled, GPUs are ranked by highest free memory, then lowest utilization, then lowest used memory.
- `min_free_mem_gb` is the free-memory threshold. If no GPU meets it, the notebook falls back to all detected GPUs (or CPU if none are available).
- `max_gpus` caps how many selected GPUs are exposed through `CUDA_VISIBLE_DEVICES`.


In [ ]:
import os
import torch

from granitewxc.utils.config import get_config
from granitewxc.utils.eccc_data import get_dataloaders_eccc
from granitewxc.models.model import get_finetune_model_UNET, get_finetune_model
from granitewxc.utils.plot import plot_sample, plot_loss

Configure the backends and torch states

In [ ]:
torch.jit.enable_onednn_fusion(True)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = True
torch.manual_seed(seed)


Select the execution device (CPU or GPU) using the hardware configuration above.


In [ ]:
if SELECTED_USE_GPU and torch.cuda.is_available():
    device = torch.device('cuda')
    use_gpu = True
else:
    device = torch.device('cpu')
    use_gpu = False


The notebook runs in a single process and uses DataParallel automatically when multiple GPUs are selected.


In [ ]:
os.chdir('/mnt/data2/kyo/granite-wxc')
local_rank, rank = 0, 0
if use_gpu and SELECTED_GPU_COUNT > 1:
    print(f"Using DataParallel with {SELECTED_GPU_COUNT} GPU(s) in this notebook session.")
else:
    print("Using single-process training (single GPU or CPU).")


## Configuration File

The model is configured using `YAML` files.

In these files, you specify:
- Paths to the input data  
- Locations of the pretrained weights  

To ensure compatibility with the provided weights during inference, keep the model configuration consistent with the original definitions 

In [ ]:
!pip install -e .

In [ ]:
config_path = './granite-geospatial-wxc-downscaling/ECCC/configs/config_UNET_small.yaml'  # default small checkpoint for <24GB GPUs
config = get_config(config_path)

## Dataloader 

In this example we will use only **1 sample of data**

To download and setup all the remaining data follow the instructions in `./examples/eccc_downscaling/notebooks/README.md`"

In [ ]:
train_dl, val_dl = get_dataloaders_eccc(config)

### Train data

The ECCC data corresponds to the entire Canadian region

For training, we adopt a strategy that randomly crops the data into patches of fixed width and height, which are then used to train the model. For inference we will use the entire region...

You can adjust the size and number of crops in the configuration file

In [ ]:
plot_sample(next(iter(train_dl)))

## Model Initialization

We provide **2** different model architectures `UNET-like` and `CONV` 

Both architectures include:  
1. **Patch Embedding**: Extracts shallow features from the input data  
2. **Feature Extraction**: Utilizes the Prithvi backbone to extract deeper features  

The key difference is that the UNET-like version incorporates **static high-resolution data** into the model

In this notebook, we use the **UNET-like** version

To switch to the **CONV** model, update the configuration file accordingly and use `get_finetune_model(config)` 

In [ ]:
model = get_finetune_model_UNET(config)

if use_gpu:
    model = model.to(device)
    if SELECTED_GPU_COUNT > 1:
        model = torch.nn.DataParallel(model, device_ids=list(range(SELECTED_GPU_COUNT)))
        print(f"Training strategy: DataParallel on {SELECTED_GPU_COUNT} GPU(s)")
    else:
        print("Training strategy: single GPU")
else:
    print("Training strategy: CPU")


We can now load the pretrained weights

In [ ]:
weights_path = config.path_model_weights
weights = torch.load(weights_path, map_location='cpu')['model']

model_state = model.state_dict()
weights_have_module_prefix = all(key.startswith('module.') for key in weights.keys())
model_expects_module_prefix = all(key.startswith('module.') for key in model_state.keys())

if model_expects_module_prefix and not weights_have_module_prefix:
    weights = weights.__class__((f"module.{key}", value) for key, value in weights.items())
elif weights_have_module_prefix and not model_expects_module_prefix:
    prefix_len = len('module.')
    weights = weights.__class__((key[prefix_len:], value) for key, value in weights.items())

model.load_state_dict(weights, strict=True)
model.to(device)


## Finetuning

The model is now ready for training

In [ ]:
from torch.optim import AdamW
from torch.cuda.amp import GradScaler
from torch.optim.lr_scheduler import CosineAnnealingLR

from granitewxc.utils.trainer import train_model
try:
    from granitewxc.models.loss import rmse_loss
except ModuleNotFoundError:
    import torch

    def rmse_loss(y_hat, y):
        return torch.sqrt(torch.mean((y_hat - y['y']) ** 2))

## 
Defining the optimizer, scaler, and scheduler for the training process.

In [ ]:
optimizer = AdamW(model.parameters(), lr=config.learning_rate)
scaler = GradScaler()
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=config.num_epochs * min(len(train_dl), config.limit_steps_train),
    eta_min=config.min_lr,
)
save_every = 5

We'll train the model using just 1 data pair and run it for only 5 epochs

Let's train it for a couple of epochs based on the configuration file

In [ ]:
train_losses, val_losses = train_model(
    config, model, train_dl, val_dl, optimizer, scheduler, scaler, local_rank, use_gpu, save_every, rmse_loss
)

In [ ]:
plot_loss(train_losses, val_losses)